# FiberNet Complete Tutorial

This tutorial demonstrates the complete workflow for fiber network research using FiberNet.

## 1. Setup and Imports

In [ ]:
import numpy as np
import fibernet as fn
from fibernet import gen
from fibernet.analysis import MorphologyAnalyzer, TopologyAnalyzer
from fibernet.sim import FiberFEM
import matplotlib.pyplot as plt

print(f"FiberNet version: {fn.__version__}")

## 2. Generating Fiber Networks

FiberNet provides many generators for different network types.

In [ ]:
# Generate a random 2D network
net_random = gen.random_straight_2d(
    num_fibers=100,
    fiber_length=15.0,
    box_size=(50, 50),
    radius=0.1,
    seed=42
)

print(f"Random network: {net_random.num_fibers} fibers, {net_random.num_crosslinks} crosslinks")

# Generate an ordered lattice
net_lattice = gen.square_lattice_2d(spacing=5.0, grid_size=(5, 5))

print(f"Lattice: {net_lattice.num_fibers} fibers, {net_lattice.num_crosslinks} crosslinks")

## 3. Structural Analysis

Analyze the morphology and topology of the network.

In [ ]:
# Morphological analysis
morph = MorphologyAnalyzer(net_random)
report = morph.full_report()

print("Morphological Properties:")
print(f"  Nematic order parameter: {report['nematic_order']:.3f}")
print(f"  Mean fiber length: {report['mean_length']:.2f}")
print(f"  Mean tortuosity: {report.get('mean_tortuosity', 1.0):.3f}")
print(f"  Volume fraction: {report['volume_fraction']:.4f}")

In [ ]:
# Topological analysis
topo = TopologyAnalyzer(net_random)
topo_report = topo.full_report()

print("\nTopological Properties:")
print(f"  Number of nodes: {topo_report['num_nodes']}")
print(f"  Number of edges: {topo_report['num_edges']}")
print(f"  Is connected: {topo_report['is_connected']}")
print(f"  Number of components: {topo_report['num_components']}")

## 4. Mechanical Simulation

Perform finite element analysis to study mechanical properties.

In [ ]:
# Create FEM solver
fem = FiberFEM(net_lattice, segments_per_fiber=5)

# Apply uniaxial strain
result = fem.apply_uniaxial_strain(strain=0.01, axis=0)

print(f"Applied strain: 1%")
print(f"Strain energy: {result.energy:.4e} J")
print(f"Max displacement: {result.max_displacement():.4e} m")
print(f"Max stress: {result.max_stress():.4e} Pa")

In [ ]:
# Compute effective modulus
E_x = fem.effective_modulus(strain=0.001, axis=0)
E_y = fem.effective_modulus(strain=0.001, axis=1)

print(f"\nEffective Young's modulus:")
print(f"  E_x = {E_x:.4e} Pa")
print(f"  E_y = {E_y:.4e} Pa")
print(f"  Anisotropy ratio: {E_x/E_y:.3f}")

## 5. Stress-Strain Curve

Generate a stress-strain curve by applying incremental strains.

In [ ]:
# Generate stress-strain data
strains = np.linspace(0, 0.05, 10)
stresses = []

for eps in strains:
    result = fem.apply_uniaxial_strain(strain=eps, axis=0)
    stress = fem.compute_effective_stress(result, axis=0)
    stresses.append(stress)

stresses = np.array(stresses)

# Plot
plt.figure(figsize=(8, 6))
plt.plot(strains, stresses, 'b-o', linewidth=2, markersize=6)
plt.xlabel('Strain', fontsize=12)
plt.ylabel('Stress (Pa)', fontsize=12)
plt.title('Stress-Strain Curve', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Network Transformations

Apply transformations to modify networks.

In [ ]:
from fibernet.core.transform import rotate, scale, mirror, merge

# Rotate by 45 degrees
net_rotated = rotate(net_random, angle=np.pi/4, axis=np.array([0, 0, 1]))

# Scale by 2x
net_scaled = scale(net_random, factor=2.0)

# Mirror along x-axis
net_mirrored = mirror(net_random, axis=0)

print("Transformations applied successfully!")
print(f"Original: {net_random.num_fibers} fibers")
print(f"Rotated: {net_rotated.num_fibers} fibers")
print(f"Scaled: {net_scaled.num_fibers} fibers")
print(f"Mirrored: {net_mirrored.num_fibers} fibers")

## 7. Specialized Generators

FiberNet includes generators for specific applications.

In [ ]:
from fibernet.gen.specialized import (
    cnt_network_2d,
    paper_network,
    textile_weave,
    electrospun_mat
)

# Carbon nanotube network
cnt_net = cnt_network_2d(
    num_tubes=50,
    tube_length=10.0,
    box_size=(40, 40),
    diameter=0.02,
    seed=42
)
print(f"CNT network: {cnt_net.num_fibers} nanotubes")

# Paper fiber network
paper_net = paper_network(
    num_fibers=100,
    fiber_length=20.0,
    box_size=(50, 50),
    curliness=0.5,
    seed=42
)
print(f"Paper network: {paper_net.num_fibers} fibers")

# Textile weave
weave_net = textile_weave(
    warp_count=10,
    weft_count=10,
    spacing=3.0,
    weave_pattern="plain"
)
print(f"Textile weave: {weave_net.num_fibers} fibers")

## 8. Machine Learning Integration

Extract features and train ML models for property prediction.

In [ ]:
from fibernet.ml.features import extract_features

# Extract features from a network
features = extract_features(net_random)

print("Extracted features:")
print(f"  Number of features: {len(features)}")
print(f"  Nematic order: {features['nematic_order']:.3f}")
print(f"  Mean length: {features['mean_length']:.2f}")
print(f"  Volume fraction: {features['volume_fraction']:.4f}")

# Show top 10 features
print("\nTop 10 features:")
for i, (key, value) in enumerate(list(features.items())[:10]):
    print(f"  {key}: {value:.4f}")

## 9. Exporting Networks

Export networks to various formats for visualization or external tools.

In [ ]:
from fibernet.io import to_vtk, to_lammps, to_xyz
import tempfile
import os

# Create temporary directory for exports
with tempfile.TemporaryDirectory() as tmpdir:
    # Export to VTK (for Paraview visualization)
    vtk_path = os.path.join(tmpdir, "network.vtk")
    to_vtk(net_random, vtk_path)
    print(f"Exported to VTK: {vtk_path}")
    
    # Export to LAMMPS (for molecular dynamics)
    lammps_path = os.path.join(tmpdir, "network.lammps")
    to_lammps(net_random, lammps_path, bead_spacing=1.0)
    print(f"Exported to LAMMPS: {lammps_path}")
    
    # Export to XYZ
    xyz_path = os.path.join(tmpdir, "network.xyz")
    to_xyz(net_random, xyz_path)
    print(f"Exported to XYZ: {xyz_path}")

print("\nAll exports completed successfully!")

## 10. High-Level API

Use the simplified high-level API for quick analysis.

In [ ]:
# Quick network creation
net = fn.create("random_2d", num_fibers=50, fiber_length=10, box_size=(30, 30))

# Quick analysis
results = fn.analyze(net)
print("Quick analysis results:")
for key, value in results.items():
    print(f"  {key}: {value}")

# Quick simulation
mech = fn.simulate_mechanics(net, strain=0.001, axis=0)
print(f"\nMechanical simulation:")
print(f"  Modulus: {mech['modulus']:.2e} Pa")

## 11. Summary

This tutorial covered:
- Network generation (random, ordered, specialized)
- Structural analysis (morphology, topology)
- Mechanical simulation (FEM, stress-strain)
- Network transformations
- Machine learning integration
- Export to various formats
- High-level API

For more information, see the [full documentation](https://fibernet.readthedocs.io/).